In [7]:
import pandas as pd
import os

raw_data_path = '../data/raw/YRBS_2007 .csv'
df = pd.read_csv(raw_data_path)

# 2. 選取本專案需要的變數
# WhatIsYourSex: 性別 (1=Female, 2=Male)
# CurrentCigaretteUse: 過去30天抽菸天數
# EverCigaretteUse: 是否曾經抽過菸 (用來做邏輯檢查)
target_cols = ['WhatIsYourSex', 'CurrentCigaretteUse', 'EverCigaretteUse']
df_subset = df[target_cols].copy()

# 3. 刪除任何含有缺失值 (NaN) 的樣本
# 這能確保我們計算比例時，分母是精確的有效回答人數
df_clean = df_subset.dropna().copy()
print(f"原始資料筆數: {len(df)}")
print(f"刪除缺失值後筆數: {len(df_clean)}")

# 4. 執行重編碼 (Recoding)
# 性別轉換：為了方便閱讀，我們通常保持原始定義或轉換為 0, 1
# 抽菸行為轉換：根據規則 success(1) = codes 2-7, failure(0) = code 1
def recode_smoking(x):
    if x >= 2 and x <= 7:
        return 1  # 有抽菸 (Success)
    elif x == 1:
        return 0  # 沒抽菸 (Failure)
    return None

df_clean['Smoking_Status'] = df_clean['CurrentCigaretteUse'].apply(recode_smoking)

# 5. 邏輯一致性檢查 (Logic Check)
# 如果 EverCigaretteUse == 2 (從未抽過), 但 Smoking_Status == 1 (現在有抽), 則為矛盾資料
conflict_condition = (df_clean['EverCigaretteUse'] == 2) & (df_clean['Smoking_Status'] == 1)
df_final = df_clean[~conflict_condition].copy()

print(f"排除邏輯矛盾樣本後，最終有效樣本數: {len(df_final)}")

# 6. 分組檢視結果 (預覽男女抽菸比例)
summary = df_final.groupby('WhatIsYourSex')['Smoking_Status'].value_counts(normalize=True).unstack()
print("\n--- 男女抽菸比例預覽 ---")
print(summary)

# 7. 儲存處理後的資料到 processed 資料夾
output_path = '../data/processed/yrbs_smoking_cleaned.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_final.to_csv(output_path, index=False)

print(f"\n檔案已儲存至: {output_path}")

原始資料筆數: 14041
刪除缺失值後筆數: 13025
排除邏輯矛盾樣本後，最終有效樣本數: 13025

--- 男女抽菸比例預覽 ---
Smoking_Status         0         1
WhatIsYourSex                     
1.0             0.825435  0.174565
2.0             0.781153  0.218847

檔案已儲存至: ../data/processed/yrbs_smoking_cleaned.csv
